# Chương 5 — TabM trên U.S. Accidents

Notebook chuyên sâu cho một pipeline: **TabM**. Mục tiêu là phân loại Severity 1–4, mức ảnh hưởng tới giao thông trong dataset; không diễn giải nhãn như số người tử vong/bị thương. Notebook dùng cùng feature protocol và split seed với notebook tổng hợp để kết quả có thể đối chiếu.

## Lộ trình thực nghiệm

1. Khảo sát CSV và kiểm tra nhãn.
2. Xây feature hợp lệ tại thời điểm bắt đầu tai nạn, bỏ cột hậu nghiệm/leakage.
3. Tạo split nhóm xấp xỉ 64/16/20; giữ cùng seed giữa các notebook.
4. Fit preprocessing trên train, huấn luyện riêng mô hình trong notebook này.
5. Chọn cấu hình bằng validation; chỉ dùng test cho báo cáo cuối.
6. Lưu config, split IDs, metrics, dự đoán và biểu đồ vào thư mục run riêng.

## Nguồn và phạm vi thuật toán

Gorishniy et al. (ICLR 2025), TabM: Advancing Tabular Deep Learning With Parameter-Efficient Ensembling, Sections 3.2–3.3. https://arxiv.org/abs/2410.24210

Ghi chú paper và quyết định triển khai nằm trong `PAPER_IMPLEMENTATION_NOTES.md`. Cấu hình notebook là DEMO để kiểm tra pipeline, không tái lập benchmark của paper.

## Cấu hình và môi trường

Chạy tuần tự từ đầu. `MODE=FULL` là mặc định và dùng toàn bộ dòng sau khi bỏ duplicate chính xác; `DEMO` lấy tối đa 30.000 dòng phân tầng với seed 217 để kiểm tra nhanh. Bốn notebook được chạy riêng, từng notebook một. Có thể đặt `DATA_PATH` nếu auto-discovery tìm nhiều CSV cùng tên. XGBoost dùng tối đa 250 cây và early stopping; mạng neural bắt đầu với tối đa 8 epoch, batch 512 và early stopping theo validation. Các mức này là cấu hình full-data khởi điểm, không phải tuning benchmark. Chỉnh ngân sách trong CONFIG nếu cần. Kaggle cần Internet nếu package chính thức còn thiếu. Với FT-Transformer/TabM nên bật GPU; CPU vẫn được hỗ trợ nhưng có thể chậm.

In [ ]:
CONFIG = {
    "MODE": "FULL",                  # DEMO hoặc FULL
    "SEED": 217,
    "DATA_PATH": None,               # None để tự tìm Accidents_500.csv
    "DEMO_N": 30000,
    "RUN_MODELS": ["TabM"],
    "WCFR_TOP_K": 10,                # cố định cho demo, không chọn trên test
    "WCFR_MAX_TRAIN_ROWS": 10000,
    "WCFR_NUMERIC_BINS": 5,
    "WCFR_MAX_CATEGORIES": 40,
    "MAX_WEATHER_AGE_MIN": 120,
    "MAX_CATEGORIES_PER_FEATURE": 128,
    "MIN_CATEGORY_COUNT": 2,
    "XGB_TREES": 250,
    "XGB_EARLY_STOPPING_ROUNDS": 20,
    "XGB_VERBOSE_EVAL": 25,
    "NEURAL_EPOCHS": 8,
    "NEURAL_PATIENCE": 3,
    "BATCH_SIZE": 512,
    "LEARNING_RATE": 0.0005,
    "WEIGHT_DECAY": 0.0001,
    "TABM_K": 8,
    "TRAIN_LOG_EVERY_STEPS": 100,
}
if CONFIG["MODE"] not in {"DEMO", "FULL"}:
    raise ValueError("MODE phải là DEMO hoặc FULL.")
if not CONFIG["RUN_MODELS"]:
    raise ValueError("RUN_MODELS cần có ít nhất một mô hình.")

import hashlib, importlib, importlib.metadata as metadata, json, os, platform
import random, subprocess, sys, time, warnings, shutil
from collections import Counter
from datetime import datetime, timezone
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import sklearn
from scipy import sparse
from sklearn.metrics import (
    accuracy_score, balanced_accuracy_score, cohen_kappa_score,
    confusion_matrix, f1_score, precision_recall_fscore_support,
)
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.preprocessing import OneHotEncoder, StandardScaler

warnings.filterwarnings("ignore", category=FutureWarning)
pd.set_option("display.max_columns", 100)
SEED = int(CONFIG["SEED"])
random.seed(SEED)
np.random.seed(SEED)
model_tag = "tabm"
RUN_ID = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ") + f"_{model_tag}_seed{SEED}"
IS_KAGGLE = Path("/kaggle/input").exists() and Path("/kaggle/working").exists()
OUTPUT_ROOT = Path("/kaggle/working/outputs") if IS_KAGGLE else Path.cwd() / "outputs"
OUTPUT_DIR = OUTPUT_ROOT / RUN_ID
OUTPUT_DIR.mkdir(parents=True, exist_ok=False)
print("Python", sys.version.split()[0], "| pandas", pd.__version__,
      "| numpy", np.__version__, "| sklearn", sklearn.__version__)
print("Kaggle:", IS_KAGGLE, "| CPU cores:", os.cpu_count())
print("Output:", OUTPUT_DIR.resolve())

### 5.2. Tìm CSV và khảo sát schema

Cell này chỉ tìm đúng tên Accidents_500.csv. Nếu nhiều bản cùng tên, cần đặt DATA_PATH thủ công. Thống kê bao gồm kích thước, kiểu, missing, duplicate, thời gian và nhãn. Checksum cho phép nhận diện đúng CSV đã dùng.

In [ ]:
def find_data_file(data_path):
    if data_path:
        path = Path(data_path).expanduser()
        if not path.is_file() or path.name.lower() != "accidents_500.csv":
            raise FileNotFoundError("DATA_PATH cần trỏ tới file Accidents_500.csv tồn tại.")
        return path.resolve()
    kaggle = Path("/kaggle/input")
    if kaggle.exists():
        matches = sorted(p for p in kaggle.rglob("Accidents_500.csv") if p.is_file())
        if len(matches) > 1:
            raise RuntimeError("Có nhiều CSV cùng tên; đặt CONFIG['DATA_PATH']:\n" +
                               "\n".join(map(str, matches)))
        if len(matches) == 1:
            return matches[0]
    local = Path.cwd() / "Accidents_500.csv"
    if local.is_file():
        return local.resolve()
    raise FileNotFoundError("Không thấy CSV. Add Input trên Kaggle hoặc đặt DATA_PATH.")

def sha256_file(path):
    digest = hashlib.sha256()
    with open(path, "rb") as handle:
        for chunk in iter(lambda: handle.read(1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()

DATA_FILE = find_data_file(CONFIG["DATA_PATH"])
DATA_SHA256 = sha256_file(DATA_FILE)
df_raw = pd.read_csv(DATA_FILE, low_memory=False)
print("CSV:", DATA_FILE, "| bytes:", DATA_FILE.stat().st_size,
      "| SHA-256:", DATA_SHA256)
print("Shape:", df_raw.shape)
if "Severity" not in df_raw or df_raw["Severity"].isna().any():
    raise ValueError("Severity thiếu hoặc có giá trị missing.")
labels_seen = sorted(pd.to_numeric(df_raw["Severity"], errors="coerce").dropna().unique())
if labels_seen != [1, 2, 3, 4]:
    raise ValueError(f"Nhãn thực tế {labels_seen}; yêu cầu đủ Severity 1–4.")

schema = pd.DataFrame({
    "column": df_raw.columns,
    "dtype": [str(df_raw[c].dtype) for c in df_raw.columns],
    "missing": [int(df_raw[c].isna().sum()) for c in df_raw.columns],
    "missing_pct": [float(df_raw[c].isna().mean() * 100) for c in df_raw.columns],
    "unique": [int(df_raw[c].nunique(dropna=True)) for c in df_raw.columns],
})
display(schema.sort_values("missing_pct", ascending=False))
label_counts = df_raw["Severity"].value_counts().sort_index()
print("Severity counts:", label_counts.to_dict())
print("Exact duplicate rows:", int(df_raw.duplicated().sum()))
print("Duplicate ID:", int(df_raw["ID"].duplicated().sum()) if "ID" in df_raw else "no ID")
for column in ["Start_Time", "End_Time", "Weather_Timestamp"]:
    if column in df_raw:
        try:
            parsed = pd.to_datetime(df_raw[column], format="mixed", errors="coerce")
        except TypeError:
            parsed = pd.to_datetime(df_raw[column], errors="coerce")
        print(column, "range:", parsed.min(), "to", parsed.max(),
              "| parse failures:", int(parsed.isna().sum()))
schema.to_csv(OUTPUT_DIR / "schema_missing_summary.csv", index=False)
pd.DataFrame({"Severity": label_counts.index, "count": label_counts.values}).to_csv(
    OUTPUT_DIR / "label_distribution_full.csv", index=False)
fig, ax = plt.subplots(figsize=(6, 3.5))
ax.bar(label_counts.index.astype(str), label_counts.values, color="#3b82f6")
ax.set(title="Phân bố Severity trong CSV", xlabel="Severity (nhãn gốc)", ylabel="Số dòng")
ax.grid(axis="y", alpha=.25)
plt.show()

### 5.3. Feature và leakage

Thời điểm dự đoán được chốt tại lúc bắt đầu/ghi nhận sự cố. Giữ tọa độ bắt đầu, khu vực, thuộc tính đường, thời điểm bắt đầu đã chuyển thành biến chu kỳ và thời tiết chỉ khi Weather_Timestamp không sau Start_Time, độ trễ tối đa 120 phút.

Bỏ ID khỏi X nhưng lưu để truy vết; bỏ Source, Airport_Code, Street, Zipcode, Country (định danh/proxy, cardinality cao hoặc hằng); bỏ Severity khỏi X; bỏ End_Time, End_Lat, End_Lng, Description, Distance(mi), Weather_Timestamp vì hậu nghiệm/nguy cơ leakage. Không đưa cột khác vào X nếu chưa thẩm định.

CSV không có event ID riêng. Split sẽ nhóm các dòng theo Start_Time và tọa độ bắt đầu làm tròn 5 chữ số; đây là chữ ký gần đúng, có thể gộp tai nạn trùng thời gian/vị trí.

In [ ]:
NUMERIC_FEATURES = [
    "start_lat", "start_lng", "start_year", "hour_sin", "hour_cos",
    "weekday_sin", "weekday_cos", "month_sin", "month_cos", "is_weekend",
    "weather_age_minutes", "temperature_f", "wind_chill_f", "humidity_pct",
    "pressure_in", "visibility_mi", "wind_speed_mph", "precipitation_in",
]
CATEGORICAL_FEATURES = [
    "city", "county", "state", "timezone", "wind_direction", "weather_condition",
    "amenity", "bump", "crossing", "give_way", "junction", "no_exit", "railway",
    "roundabout", "station", "stop", "traffic_calming", "traffic_signal",
    "turning_loop", "sunrise_sunset", "civil_twilight", "nautical_twilight",
    "astronomical_twilight",
]
FEATURE_NAMES = NUMERIC_FEATURES + CATEGORICAL_FEATURES
TARGET_TO_CODE = {1: 0, 2: 1, 3: 2, 4: 3}
CODE_TO_TARGET = {v: k for k, v in TARGET_TO_CODE.items()}
CLASS_CODES = np.arange(4, dtype=np.int64)
ROAD_MAP = {
    "amenity":"Amenity", "bump":"Bump", "crossing":"Crossing",
    "give_way":"Give_Way", "junction":"Junction", "no_exit":"No_Exit",
    "railway":"Railway", "roundabout":"Roundabout", "station":"Station",
    "stop":"Stop", "traffic_calming":"Traffic_Calming",
    "traffic_signal":"Traffic_Signal", "turning_loop":"Turning_Loop",
}
NUM_SOURCE = {
    "temperature_f":"Temperature(F)", "wind_chill_f":"Wind_Chill(F)",
    "humidity_pct":"Humidity(%)", "pressure_in":"Pressure(in)",
    "visibility_mi":"Visibility(mi)", "wind_speed_mph":"Wind_Speed(mph)",
    "precipitation_in":"Precipitation(in)",
}
CAT_SOURCE = {
    "city":"City", "county":"County", "state":"State", "timezone":"Timezone",
    "wind_direction":"Wind_Direction", "weather_condition":"Weather_Condition",
    "sunrise_sunset":"Sunrise_Sunset", "civil_twilight":"Civil_Twilight",
    "nautical_twilight":"Nautical_Twilight", "astronomical_twilight":"Astronomical_Twilight",
    **ROAD_MAP,
}

df = df_raw.copy()
df["_source_row"] = np.arange(len(df), dtype=np.int64)
df = df.drop_duplicates(subset=list(df_raw.columns), keep="first").reset_index(drop=True)

if CONFIG["MODE"] == "DEMO":
    if len(df) < CONFIG["DEMO_N"]:
        raise ValueError("DEMO_N lớn hơn số dòng còn lại.")
    sampled, _ = sklearn.model_selection.train_test_split(
        df, train_size=int(CONFIG["DEMO_N"]), random_state=SEED,
        stratify=df["Severity"].astype(int),
    )
    df_run = sampled.sort_values("_source_row").reset_index(drop=True)
else:
    df_run = df.reset_index(drop=True)

def make_safe_features(frame):
    try:
        start = pd.to_datetime(frame.get("Start_Time"), format="mixed", errors="coerce")
        weather = pd.to_datetime(
            frame.get("Weather_Timestamp", pd.Series(pd.NaT, index=frame.index)),
            format="mixed", errors="coerce",
        )
    except TypeError:
        start = pd.to_datetime(frame.get("Start_Time"), errors="coerce")
        weather = pd.to_datetime(
            frame.get("Weather_Timestamp", pd.Series(pd.NaT, index=frame.index)),
            errors="coerce",
        )
    age = (start - weather).dt.total_seconds() / 60
    weather_ok = age.ge(0) & age.le(float(CONFIG["MAX_WEATHER_AGE_MIN"]))
    hour = start.dt.hour + start.dt.minute.fillna(0) / 60
    weekday, month = start.dt.dayofweek, start.dt.month
    x = pd.DataFrame(index=frame.index)
    x["start_lat"] = pd.to_numeric(frame.get("Start_Lat"), errors="coerce")
    x["start_lng"] = pd.to_numeric(frame.get("Start_Lng"), errors="coerce")
    x["start_year"] = start.dt.year.astype(float)
    x["hour_sin"], x["hour_cos"] = np.sin(2*np.pi*hour/24), np.cos(2*np.pi*hour/24)
    x["weekday_sin"], x["weekday_cos"] = np.sin(2*np.pi*weekday/7), np.cos(2*np.pi*weekday/7)
    x["month_sin"], x["month_cos"] = np.sin(2*np.pi*(month-1)/12), np.cos(2*np.pi*(month-1)/12)
    x["is_weekend"] = weekday.isin([5, 6]).astype(float)
    x["weather_age_minutes"] = age.where(weather_ok)
    for out, source in NUM_SOURCE.items():
        x[out] = pd.to_numeric(frame[source], errors="coerce").where(weather_ok) if source in frame else np.nan
    for out, source in CAT_SOURCE.items():
        if source in frame:
            values = frame[source].astype("string").str.strip()
            if out in {"wind_direction", "weather_condition"}:
                values = values.where(weather_ok, pd.NA)
            x[out] = values
        else:
            x[out] = pd.Series(pd.NA, index=frame.index, dtype="string")
    for col in NUMERIC_FEATURES:
        x[col] = pd.to_numeric(x[col], errors="coerce")
    for col in CATEGORICAL_FEATURES:
        x[col] = x[col].astype("string").str.strip().replace("", pd.NA).fillna("__MISSING__").astype(str)
    return x[FEATURE_NAMES]

X = make_safe_features(df_run)
y_original = df_run["Severity"].astype(int).to_numpy()
y = np.array([TARGET_TO_CODE[int(v)] for v in y_original], dtype=np.int64)
assert "Severity" not in X.columns and not ({"ID", "End_Time", "Description"} & set(X.columns))
if set(np.unique(y)) != {0, 1, 2, 3}:
    raise ValueError("Sau DEMO sampling không còn đủ lớp; tăng DEMO_N hoặc dùng FULL.")
print("MODE:", CONFIG["MODE"], "| rows:", X.shape[0], "| features:", X.shape[1])
print("DEMO class counts:", pd.Series(y_original).value_counts().sort_index().to_dict())
display(X.head(3))

### 5.3. Split chung, tránh giao cắt nhóm

Dùng StratifiedGroupKFold hai lượt: fold ngoài xấp xỉ 20% làm test; một fold trong phần còn lại xấp xỉ 16% tổng làm validation; phần còn lại xấp xỉ 64% làm train. Split được dùng cho cả bốn phương pháp và ID được lưu. Lớp hiếm phải đạt ngưỡng tối thiểu trong từng tập, nếu không notebook dừng để tăng sample thay vì bỏ lớp.

In [ ]:
def event_groups(frame):
    try:
        start = pd.to_datetime(frame.get("Start_Time"), format="mixed", errors="coerce")
    except TypeError:
        start = pd.to_datetime(frame.get("Start_Time"), errors="coerce")
    lat, lng = pd.to_numeric(frame.get("Start_Lat"), errors="coerce"), pd.to_numeric(frame.get("Start_Lng"), errors="coerce")
    valid = start.notna() & lat.notna() & lng.notna()
    result = pd.Series(index=frame.index, dtype="string")
    result.loc[valid] = (
        start.loc[valid].dt.strftime("%Y-%m-%d %H:%M:%S") + "|"
        + lat.loc[valid].round(5).map(lambda z: f"{z:.5f}") + "|"
        + lng.loc[valid].round(5).map(lambda z: f"{z:.5f}")
    )
    fallback = frame.loc[~valid, "ID"].astype(str) if "ID" in frame else frame.loc[~valid, "_source_row"].astype(str)
    result.loc[~valid] = "unkeyed|" + fallback
    return result.astype(str).to_numpy()

groups = event_groups(df_run)
outer = StratifiedGroupKFold(5, shuffle=True, random_state=SEED)
trainval_pos, test_pos = next(outer.split(np.zeros(len(y)), y, groups))
inner = StratifiedGroupKFold(5, shuffle=True, random_state=SEED+1)
train_local, val_local = next(inner.split(
    np.zeros(len(trainval_pos)), y[trainval_pos], groups[trainval_pos]))
train_pos, val_pos = trainval_pos[train_local], trainval_pos[val_local]

def check_counts(name, values, minimum):
    counts = pd.Series(values).value_counts().reindex(CLASS_CODES, fill_value=0)
    readable = {CODE_TO_TARGET[i]: int(counts[i]) for i in CLASS_CODES}
    print(name, "n =", len(values), "| Severity =", readable)
    if (counts < minimum).any():
        raise ValueError(f"{name} có lớp dưới ngưỡng {minimum}: {readable}. Tăng DEMO_N hoặc chọn FULL.")

check_counts("Train", y[train_pos], 20)
check_counts("Validation", y[val_pos], 10)
check_counts("Test", y[test_pos], 10)
g_train, g_val, g_test = set(groups[train_pos]), set(groups[val_pos]), set(groups[test_pos])
assert not (g_train & g_val or g_train & g_test or g_val & g_test)

SPLIT = {
    "train": (X.iloc[train_pos].reset_index(drop=True), y[train_pos]),
    "validation": (X.iloc[val_pos].reset_index(drop=True), y[val_pos]),
    "test": (X.iloc[test_pos].reset_index(drop=True), y[test_pos]),
}
split_name = np.empty(len(df_run), dtype=object)
split_name[train_pos], split_name[val_pos], split_name[test_pos] = "train", "validation", "test"
pd.DataFrame({
    "split": split_name, "source_row_index": df_run["_source_row"].to_numpy(),
    "ID": df_run["ID"].to_numpy() if "ID" in df_run else df_run["_source_row"].to_numpy(),
    "Severity": y_original, "event_group_approx": groups,
}).to_csv(OUTPUT_DIR / "split_ids.csv", index=False)
with open(OUTPUT_DIR / "class_mapping.json", "w", encoding="utf-8") as f:
    json.dump({"original_to_encoded": {str(k):v for k,v in TARGET_TO_CODE.items()},
               "encoded_to_original": {str(k):v for k,v in CODE_TO_TARGET.items()}}, f, indent=2)
print("Split ratio:", {k: round(len(v[1])/len(y), 4) for k,v in SPLIT.items()})
group_table = pd.DataFrame({"group": groups, "severity": y_original}).groupby("group").agg(
    rows=("severity","size"), labels=("severity","nunique"))
print("Repeated approximate groups:", int((group_table.rows > 1).sum()),
      "| groups with multiple labels:", int((group_table.labels > 1).sum()))

## 5.4. Tiền xử lý riêng cho kiến trúc này

Median, vocabulary/category codes, discretization cần thiết và class weights chỉ được fit bằng train. XGBoost dùng sparse one-hot; mạng neural nhận numerical values đã scale cùng missing indicators và categorical integer codes. Cột unseen/rare đi vào mã 0. Các pipeline dưới đây dùng cùng feature hợp lệ và split.

In [ ]:
preprocessing_start = time.perf_counter()
X_train, y_train = SPLIT["train"]
X_val, y_val = SPLIT["validation"]
X_test, y_test = SPLIT["test"]

# Median, vocabulary, scaler và encoder chỉ fit trên train.
numeric_medians = X_train[NUMERIC_FEATURES].median().fillna(0.0)
n_num = len(NUMERIC_FEATURES)
def numeric_block(frame):
    values = frame[NUMERIC_FEATURES].apply(pd.to_numeric, errors="coerce").to_numpy(np.float32)
    missing = np.isnan(values).astype(np.float32)
    values = np.where(np.isnan(values), numeric_medians.to_numpy(np.float32)[None,:], values)
    return np.concatenate([values.astype(np.float32), missing], axis=1)

cat_vocabs, cat_cards = {}, []
for col in CATEGORICAL_FEATURES:
    counts = X_train[col].astype(str).value_counts()
    keep = [v for v,n in counts.items() if n >= CONFIG["MIN_CATEGORY_COUNT"] and v != "__MISSING__"]
    keep = sorted(keep, key=lambda v: (-int(counts[v]), str(v)))[:CONFIG["MAX_CATEGORIES_PER_FEATURE"]-2]
    vocab = {v:i+1 for i,v in enumerate(keep)}
    if (X_train[col].astype(str) == "__MISSING__").any():
        vocab["__MISSING__"] = len(vocab)+1
    cat_vocabs[col] = vocab
    cat_cards.append(len(vocab)+1)

def cat_block(frame):
    return np.column_stack([
        frame[col].astype(str).map(cat_vocabs[col]).fillna(0).astype(np.int64).to_numpy()
        for col in CATEGORICAL_FEATURES
    ]).astype(np.int64)

num_train, num_val, num_test = numeric_block(X_train), numeric_block(X_val), numeric_block(X_test)
train_cat, val_cat, test_cat = cat_block(X_train), cat_block(X_val), cat_block(X_test)
scaler = StandardScaler().fit(num_train)
train_num_nn = scaler.transform(num_train).astype(np.float32)
val_num_nn = scaler.transform(num_val).astype(np.float32)
test_num_nn = scaler.transform(num_test).astype(np.float32)
X_train_xgb = X_val_xgb = X_test_xgb = None
print("Neural numerical/categorical:", train_num_nn.shape, train_cat.shape,
      "| cardinalities:", cat_cards)

class_counts = np.bincount(y_train, minlength=4)
if (class_counts == 0).any():
    raise ValueError("Train thiếu lớp.")
class_weights = len(y_train) / (4.0 * class_counts.astype(np.float64))
print("Class weights train:", {CODE_TO_TARGET[i]: round(float(class_weights[i]),4) for i in range(4)})
PREPROCESSING_SECONDS = time.perf_counter() - preprocessing_start
print("Preprocessing seconds:", round(PREPROCESSING_SECONDS, 2))


## 5.5. Metrics và kiểm soát output

Metric chính là Macro-F1; báo thêm Accuracy, Weighted-F1, Balanced Accuracy, QWK, metric từng lớp và confusion matrix với thứ tự nhãn cố định 1–4. QWK xét khoảng cách thứ tự nhãn nhưng Severity vẫn là mức ảnh hưởng giao thông. Validation dùng để chọn checkpoint/điểm dừng; test chỉ được chấm sau khi chọn xong. Không có score 0 giả cho nhánh lỗi.

In [ ]:
METRIC_COLS = ["run_id","mode","model","n_test","macro_f1","accuracy","weighted_f1",
               "balanced_accuracy","qwk","preprocessing_seconds","feature_selection_seconds",
               "training_seconds","inference_seconds"]
CLASS_COLS = ["run_id","mode","model","Severity","precision","recall","f1","support"]
metric_rows, class_rows, pred_frames, status_rows, histories, execution_devices = [], [], [], [], {}, {}
selected_features = []

def check_probabilities(prob, n):
    prob = np.asarray(prob, dtype=np.float64)
    if prob.shape != (n,4) or not np.isfinite(prob).all():
        raise ValueError(f"Xác suất cần shape ({n},4) và hữu hạn, nhận {prob.shape}.")
    if not np.allclose(prob.sum(axis=1), 1.0, atol=1e-4):
        raise ValueError("Tổng xác suất từng dòng không bằng 1.")
    return prob

def add_success(name, prob, training=0., inference=0., selection=0., preprocessing=0.):
    prob = check_probabilities(prob, len(y_test))
    pred = prob.argmax(1)
    metric_rows.append({
        "run_id":RUN_ID,"mode":CONFIG["MODE"],"model":name,"n_test":len(y_test),
        "macro_f1":f1_score(y_test,pred,labels=CLASS_CODES,average="macro",zero_division=0),
        "accuracy":accuracy_score(y_test,pred),
        "weighted_f1":f1_score(y_test,pred,labels=CLASS_CODES,average="weighted",zero_division=0),
        "balanced_accuracy":balanced_accuracy_score(y_test,pred),
        "qwk":cohen_kappa_score(y_test,pred,labels=CLASS_CODES,weights="quadratic"),
        "preprocessing_seconds":preprocessing,"feature_selection_seconds":selection,
        "training_seconds":training,"inference_seconds":inference})
    p,r,f,s = precision_recall_fscore_support(y_test,pred,labels=CLASS_CODES,zero_division=0)
    for i in range(4):
        class_rows.append({"run_id":RUN_ID,"mode":CONFIG["MODE"],"model":name,
                           "Severity":CODE_TO_TARGET[i],"precision":p[i],"recall":r[i],
                           "f1":f[i],"support":int(s[i])})
    out = pd.DataFrame({
        "run_id":RUN_ID,"mode":CONFIG["MODE"],"model":name,
        "ID":df_run.iloc[test_pos]["ID"].to_numpy() if "ID" in df_run else df_run.iloc[test_pos]["_source_row"].to_numpy(),
        "true_severity":[CODE_TO_TARGET[int(v)] for v in y_test],
        "predicted_severity":[CODE_TO_TARGET[int(v)] for v in pred]})
    for i in range(4): out[f"prob_severity_{i+1}"] = prob[:,i]
    pred_frames.append(out)
    status_rows.append({"run_id":RUN_ID,"mode":CONFIG["MODE"],"model":name,"status":"success","message":""})
    matrix = confusion_matrix(y_test,pred,labels=CLASS_CODES)
    fig, ax = plt.subplots(figsize=(5,4))
    image = ax.imshow(matrix,cmap="Blues")
    ax.set(title=f"Confusion — {name} ({CONFIG['MODE']})",xlabel="Dự đoán Severity",
           ylabel="Thực tế Severity",xticks=range(4),yticks=range(4),
           xticklabels=[1,2,3,4],yticklabels=[1,2,3,4])
    for row in range(4):
        for col in range(4):
            ax.text(col,row,str(matrix[row,col]),ha="center",va="center")
    fig.colorbar(image,ax=ax); fig.tight_layout()
    fig.savefig(OUTPUT_DIR / f"confusion_{name.lower().replace('-','_')}.png",dpi=140)
    plt.show()

def add_failure(name, error):
    status_rows.append({"run_id":RUN_ID,"mode":CONFIG["MODE"],"model":name,
                        "status":"failed","message":f"{type(error).__name__}: {error}"})
    print(name,"failed:",f"{type(error).__name__}: {error}")

def ensure_package(import_name, distribution):
    try:
        return importlib.import_module(import_name)
    except ImportError:
        print("Cài package còn thiếu:", distribution)
        subprocess.check_call([sys.executable,"-m","pip","install","--quiet",distribution])
        importlib.invalidate_caches()
        return importlib.import_module(import_name)

def package_version(distribution):
    try: return metadata.version(distribution)
    except metadata.PackageNotFoundError: return "not-installed"

def sync_torch():
    t = sys.modules.get("torch")
    if t is not None and t.cuda.is_available(): t.cuda.synchronize()

def release_model_memory():
    import gc
    globals().pop("model", None)
    globals().pop("logits", None)
    gc.collect()
    t = sys.modules.get("torch")
    if t is not None and t.cuda.is_available():
        t.cuda.empty_cache()

## 5.6. TabM: nhiều MLP song song với trọng số chia sẻ

**Nguồn:** Gorishniy et al. (ICLR 2025), Sections 3.2–3.3; package chính thức `tabm` (bản 0.0.3 nếu cần cài). Notebook dùng variant `tabm` chuẩn qua `TabM.make`, không thay bằng MLP thông thường, TabM-mini hay TabM†.

### Cấu trúc và shape

Preprocessing tạo numerical tensor `B × 36` (18 giá trị đã median-impute/scale cộng 18 missing indicators) và categorical codes `B × 23`; vocabulary chỉ fit trên train, code 0 dành cho category rare/unseen. `TabM.make(..., arch_type="tabm", k=8)` tạo 8 thành viên MLP song song. Thành viên chia sẻ phần lớn weights, còn các tham số riêng/adapter tạo khác biệt giữa các dự đoán mà tránh chi phí tám mạng độc lập.

Với batch size `B`, forward phải trả logits `B × K × 4` (`K=8`). Notebook kiểm tra shape và tính hữu hạn trước khi train. Trục cuối là Severity code 0–3; trục giữa định danh member.

### Training objective khác inference aggregation

Cho member `k`, `CE_k = CrossEntropy(logits[:, k, :], y; class_weight)`. Loss được tính riêng cho mỗi member và lấy trung bình:

`L = (1/K) * sum_{k=1..K} CE_k`.

Gradient vì vậy học từng nhánh dự báo; không tính loss trên logits đã gộp. Dùng AdamW, class weights lấy từ train, và cùng batch/epoch budget với FT demo. Validation Macro-F1 được tính từ xác suất gộp để chọn checkpoint tốt nhất.

Ở inference, với member `k`, tính `p_k = softmax(logits[:, k, :])`, rồi `p = (1/K) * sum_k p_k`. Dự đoán lớp là `argmax(p)`. Softmax từng member rồi trung bình xác suất là quy tắc của phương pháp; trung bình logits trước softmax có thể cho xác suất khác.

### Cấu hình và diễn giải

Lần chạy mặc định dùng toàn bộ train split, `K=8`, 2 blocks, chiều ẩn 128, dropout 0.1, tối đa 8 epochs, batch 512; paper thường khảo sát ngân sách/kiến trúc khác và không được tái lập bởi cấu hình này. Notebook in loss theo batch mỗi 100 bước (và ở batch đầu/cuối), tổng hợp theo epoch, rồi ghi `training_progress_tabm.csv`. Xem loss train (trung bình CE member), validation Macro-F1 (chọn checkpoint), rồi test Macro-F1 và confusion matrix (báo cáo cuối). Các đầu ra xác suất chỉ mô tả phân lớp Severity; nhãn không phải số thương vong.

### Mạng helper dùng chung

`predict_neural` chạy inference theo batch. `train_neural` tính CrossEntropyLoss từng member rồi trung bình, theo dõi validation Macro-F1 và khôi phục checkpoint tốt nhất. Inference softmax từng member rồi trung bình xác suất; training và inference aggregation được giữ riêng đúng quy trình.

In [ ]:
def predict_neural(model, num_array, cat_array, device, tabm=False):
    import torch
    model.eval(); result=[]
    with torch.no_grad():
        for start in range(0,len(num_array),CONFIG["BATCH_SIZE"]):
            end=min(start+CONFIG["BATCH_SIZE"],len(num_array))
            xn=torch.as_tensor(num_array[start:end],dtype=torch.float32,device=device)
            xc=torch.as_tensor(cat_array[start:end],dtype=torch.long,device=device)
            logits=model(xn,xc)
            prob=torch.softmax(logits,dim=-1).mean(dim=1) if tabm else torch.softmax(logits,dim=-1)
            result.append(prob.cpu().numpy())
    return np.concatenate(result)

def train_neural(model,name,tabm=False):
    import copy, torch
    import torch.nn.functional as F
    from torch.utils.data import DataLoader,TensorDataset
    device=torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model.to(device)
    if device.type=="cpu" and CONFIG["MODE"]=="FULL":
        print("Lưu ý: đang train FULL trên CPU; FT-Transformer/TabM có thể mất nhiều thời gian.",flush=True)
    weights=torch.as_tensor(class_weights,dtype=torch.float32,device=device)
    dataset=TensorDataset(torch.as_tensor(train_num_nn),torch.as_tensor(train_cat),torch.as_tensor(y_train))
    loader=DataLoader(dataset,batch_size=CONFIG["BATCH_SIZE"],shuffle=True,num_workers=0,
                      generator=torch.Generator().manual_seed(SEED),pin_memory=device.type=="cuda")
    if name=="FT-Transformer":
        optimizer=torch.optim.AdamW(model.make_parameter_groups(),lr=CONFIG["LEARNING_RATE"],weight_decay=CONFIG["WEIGHT_DECAY"])
    else:
        optimizer=torch.optim.AdamW(model.parameters(),lr=CONFIG["LEARNING_RATE"],weight_decay=CONFIG["WEIGHT_DECAY"])
    best,best_state,best_epoch=-np.inf,None,None; patience=CONFIG["NEURAL_PATIENCE"]; log=[]; batch_progress=[]
    total_batches=len(loader)
    print(name,"train rows",len(dataset),"batch size",CONFIG["BATCH_SIZE"],"batches/epoch",total_batches,"device",device,flush=True)
    for epoch in range(1,CONFIG["NEURAL_EPOCHS"]+1):
        model.train(); losses=[]
        epoch_start=time.perf_counter()
        for batch_index,(xn,xc,yy) in enumerate(loader,start=1):
            non_blocking=device.type=="cuda"
            xn,xc,yy=xn.to(device,non_blocking=non_blocking),xc.to(device,non_blocking=non_blocking),yy.to(device,non_blocking=non_blocking)
            optimizer.zero_grad(set_to_none=True); logits=model(xn,xc)
            if tabm:
                if logits.ndim!=3 or logits.shape[0]!=len(yy) or logits.shape[2]!=4:
                    raise ValueError(f"TabM cần B×k×4, nhận {tuple(logits.shape)}")
                loss=torch.stack([F.cross_entropy(logits[:,j,:],yy,weight=weights,reduction="mean")
                                  for j in range(logits.shape[1])]).mean()
            else:
                if tuple(logits.shape)!=(len(yy),4): raise ValueError(f"FT logits sai: {tuple(logits.shape)}")
                loss=F.cross_entropy(logits,yy,weight=weights,reduction="mean")
            if not torch.isfinite(loss): raise FloatingPointError("Loss không hữu hạn.")
            loss.backward(); optimizer.step()
            losses.append(loss.detach())
            should_log=(batch_index==1 or batch_index%int(CONFIG["TRAIN_LOG_EVERY_STEPS"])==0 or batch_index==total_batches)
            if should_log:
                batch_loss=float(loss.detach().cpu())
                elapsed=time.perf_counter()-epoch_start
                batch_progress.append({"epoch":epoch,"batch":batch_index,"batches":total_batches,
                                       "weighted_train_cross_entropy":batch_loss,"elapsed_seconds":elapsed})
                print(name,"epoch",epoch,"batch",f"{batch_index}/{total_batches}",
                      "weighted train CE",round(batch_loss,4),"elapsed s",round(elapsed,1),flush=True)
        val_prob=predict_neural(model,val_num_nn,val_cat,device,tabm)
        score=f1_score(y_val,val_prob.argmax(1),labels=CLASS_CODES,average="macro",zero_division=0)
        log.append({"epoch":epoch,"weighted_train_cross_entropy":float(torch.stack(losses).mean().cpu()),
                    "validation_macro_f1":float(score),"epoch_seconds":float(time.perf_counter()-epoch_start)})
        print(name,"epoch",epoch,"train weighted CE",round(float(torch.stack(losses).mean().cpu()),4),"val macro-F1",round(float(score),4),"epoch seconds",round(time.perf_counter()-epoch_start,1),flush=True)
        if score>best+1e-6:
            best=float(score); best_epoch=epoch
            best_state={k:v.detach().cpu().clone() for k,v in model.state_dict().items()}
            patience=CONFIG["NEURAL_PATIENCE"]
        else:
            patience-=1
            if patience<=0: break
    if best_state is None: raise RuntimeError("Không có neural checkpoint.")
    model.load_state_dict(best_state); model.to(device)
    histories[name]=pd.DataFrame(log)
    histories[name].to_csv(OUTPUT_DIR/f"learning_curve_{name.lower().replace('-','_')}.csv",index=False)
    pd.DataFrame(batch_progress).to_csv(OUTPUT_DIR/f"training_progress_{name.lower().replace('-','_')}.csv",index=False)
    return model,device,best_epoch,best

def fit_neural_and_score(name,model,tabm=False):
    global model_training_seconds
    start=time.perf_counter()
    # Training timer ends after checkpoint selection, before test inference.
    model,device,best_epoch,best_val=train_neural(model,name,tabm)
    sync_torch(); model_training_seconds=time.perf_counter()-start
    sync_torch()
    predict_neural(model,test_num_nn[:min(32,len(y_test))],test_cat[:min(32,len(y_test))],device,tabm)
    sync_torch(); start=time.perf_counter()
    prob=predict_neural(model,test_num_nn,test_cat,device,tabm); sync_torch()
    inference=time.perf_counter()-start
    execution_devices[name] = {
        "training":str(device), "inference":str(device),
        "input_format":"PyTorch tensors", "batch_size":int(CONFIG["BATCH_SIZE"]),
    }
    add_success(name,prob,training=model_training_seconds,inference=inference,
                preprocessing=PREPROCESSING_SECONDS)
    print(name,"best epoch",best_epoch,"validation Macro-F1",round(best_val,4),"device",device)
    return model

### Khởi tạo TabM chuẩn, kiểm tra B × K × 4

In [ ]:
if "TabM" in CONFIG["RUN_MODELS"]:
    try:
        ensure_package("tabm","tabm==0.0.3")
        from tabm import TabM
        import torch
        model=TabM.make(
            n_num_features=train_num_nn.shape[1],cat_cardinalities=cat_cards,d_out=4,
            arch_type="tabm",k=int(CONFIG["TABM_K"]),n_blocks=2,d_block=128,dropout=0.1)
        with torch.no_grad():
            logits=model(torch.as_tensor(train_num_nn[:3]),torch.as_tensor(train_cat[:3]))
        expected=(3,int(CONFIG["TABM_K"]),4)
        if tuple(logits.shape)!=expected or not torch.isfinite(logits).all():
            raise ValueError(f"Forward TabM cần {expected}, nhận {tuple(logits.shape)}")
        fit_neural_and_score("TabM",model,True)
        del logits
    except Exception as e: add_failure("TabM",e)
    finally: release_model_memory()

## 5.7. Báo cáo và lưu run

Metric test chỉ được tạo nếu mô hình trả xác suất hữu hạn với shape `N_test × 4`, tổng xác suất mỗi dòng bằng 1. Output chứa trạng thái, dự đoán/xác suất, metric tổng hợp và theo lớp, confusion matrix, cấu hình, versions/hardware, split IDs và mapping Severity. Mạng neural có thêm learning curve. Với WCFR có ranking và fit configuration của selector.

In [ ]:
metrics=pd.DataFrame(metric_rows,columns=METRIC_COLS)
class_metrics=pd.DataFrame(class_rows,columns=CLASS_COLS)
statuses=pd.DataFrame(status_rows,columns=["run_id","mode","model","status","message"])
predictions=pd.concat(pred_frames,ignore_index=True) if pred_frames else pd.DataFrame(
    columns=["run_id","mode","model","ID","true_severity","predicted_severity",
             "prob_severity_1","prob_severity_2","prob_severity_3","prob_severity_4"])
metrics.to_csv(OUTPUT_DIR/"metrics.csv",index=False)
class_metrics.to_csv(OUTPUT_DIR/"class_metrics.csv",index=False)
statuses.to_csv(OUTPUT_DIR/"model_status.csv",index=False)
predictions.to_csv(OUTPUT_DIR/"predictions.csv",index=False)
if len(metrics):
    display(metrics.sort_values("macro_f1",ascending=False))
    display(class_metrics.sort_values(["model","Severity"]))
    view=metrics.sort_values("macro_f1")
    fig,ax=plt.subplots(figsize=(8,max(3.5,.65*len(view))))
    ax.barh(view.model,view.macro_f1,color="#0f766e")
    ax.set(title=f"Macro-F1 test — {CONFIG['MODE']}",xlabel="Macro-F1",xlim=(0,1))
    ax.grid(axis="x",alpha=.25); fig.tight_layout()
    fig.savefig(OUTPUT_DIR/"test_macro_f1.png",dpi=150); plt.show()
else:
    print("Chưa có mô hình tạo metric; xem model_status.csv.")
if histories:
    fig,axes=plt.subplots(len(histories),2,figsize=(10,3.5*len(histories)),squeeze=False)
    for row,(name,h) in enumerate(histories.items()):
        axes[row,0].plot(h.epoch,h.weighted_train_cross_entropy,marker="o")
        axes[row,0].set(title=name+" weighted train CE",xlabel="Epoch",ylabel="Loss")
        axes[row,1].plot(h.epoch,h.validation_macro_f1,marker="o",color="#be123c")
        axes[row,1].set(title=name+" validation Macro-F1",xlabel="Epoch",ylabel="Macro-F1")
    fig.tight_layout(); fig.savefig(OUTPUT_DIR/"learning_curves.png",dpi=150); plt.show()

with open(OUTPUT_DIR/"config.json","w",encoding="utf-8") as f:
    json.dump({**CONFIG,"run_id":RUN_ID,"data_file_name":DATA_FILE.name,
        "data_sha256":DATA_SHA256,"actual_rows_before_sample":len(df),
        "actual_rows_used":len(df_run),"features":FEATURE_NAMES,
        "split_rows":{k:len(v[1]) for k,v in SPLIT.items()},
        "class_weights_from_train":{str(CODE_TO_TARGET[i]):float(class_weights[i]) for i in range(4)},
        "wcfr_selected_features":selected_features},f,ensure_ascii=False,indent=2)
torch=sys.modules.get("torch")
gpu=bool(torch is not None and torch.cuda.is_available())
environment={"python":sys.version,"platform":platform.platform(),"processor":platform.processor(),
    "logical_cpu_count":os.cpu_count(),"gpu_available_for_torch":gpu,
    "gpu_name":torch.cuda.get_device_name(0) if gpu else None,
    "model_execution_devices":execution_devices,
    "packages":{"numpy":np.__version__,"pandas":pd.__version__,"scikit-learn":sklearn.__version__,
      "xgboost":package_version("xgboost"),"torch":package_version("torch"),
      "rtdl_revisiting_models":package_version("rtdl_revisiting_models"),"tabm":package_version("tabm")}}
with open(OUTPUT_DIR/"environment.json","w",encoding="utf-8") as f:
    json.dump(environment,f,ensure_ascii=False,indent=2)
display(statuses)

### Hạn chế và cách tiếp tục

Kết quả được gắn nhãn theo MODE trong config. Cả DEMO và FULL đều dùng split ngẫu nhiên có nhóm gần đúng, không phải đánh giá triển khai tương lai; temporal holdout là phân tích riêng. Weather chỉ được dùng khi timestamp không sau Start_Time và tuổi tối đa 120 phút—đây là giả định thời điểm dự đoán cần bảo vệ trong báo cáo. WCFR dùng plug-in MI/CMI và lượng hóa train-only do nhóm chọn. Không diễn giải một lần chạy thành kết luận thuật toán luôn tốt hơn.

Output được tạo tại `outputs/<run_id>` (local) hoặc `/kaggle/working/outputs/<run_id>` (Kaggle). Cell cuối tạo ZIP riêng cho mô hình này.

In [ ]:
zip_path=Path(shutil.make_archive(str(OUTPUT_ROOT/f"Chapter5_{model_tag}_{RUN_ID}"),"zip",root_dir=OUTPUT_DIR))
print("Output directory:",OUTPUT_DIR.resolve())
print("ZIP:",zip_path.resolve())
try:
    from IPython.display import FileLink, display as ipy_display
    ipy_display(FileLink(str(zip_path)))
except Exception:
    pass